# EIG-Based Clarification for Code Generation

The model writes several candidate programs, asks up to 4 yes/no questions (unit tests)
chosen by Expected Information Gain, and then submits the best candidate.
Based on "20 Questions for Code" (Garg, Kim, Xi).

Code is in `generate.py`, `evaluator.py`, `scoring.py`, `clarification.py`.
This notebook runs the experiments and shows the results.

**Runs**
- 1.5B: MBPP one-shot, MBPP EIG
- 3B: MBPP EIG, MBPP+ EIG

To change the model, edit `MODEL_NAME` in `generate.py` and restart the kernel.

## Setup
Install the packages with `pip install -r requirements.txt` first.
Set `PROJECT_DIR` to the folder that has the `.py` files.

In [ ]:
import os, sys

PROJECT_DIR = "."
os.chdir(PROJECT_DIR)
print("working folder:", os.getcwd())

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # stops a warning
sys.path.append(".")

import json, math
import torch
from tqdm import tqdm
from datasets import load_dataset

from clarification import run_clarification_pipeline, run_one_shot
from evaluator import evaluate_final
from generate import (MODEL_NAME, SHOW_EXAMPLE, N_CANDIDATES, M_TESTS, K_MAX,
                      EPS, GAMMA, ETA, LAMBDA, RHO, MIN_COVERAGE)

print(f"MODEL={MODEL_NAME}  SHOW_EXAMPLE={SHOW_EXAMPLE}")
print(f"N_CANDIDATES={N_CANDIDATES}  M_TESTS={M_TESTS}  K_MAX={K_MAX}")

## Settings for the quick test
Only used by the check and quick test cells.

In [ ]:
DATASET = "mbpp"      # "mbpp" or "mbppplus"
METHOD = "oneshot"    # "oneshot" or "eig"
SPLIT = "test"        # "validation" for trying settings (mbpp only)
N_PROBLEMS = 5

## Load data

In [ ]:
# load a dataset in one common format
def load_problems(name, split):
    problems = []
    if name == "mbpp":
        # sanitized = hand-checked version of MBPP
        data = load_dataset("google-research-datasets/mbpp", "sanitized")[split]
        for s in data:
            problems.append({
                "task_id": s["task_id"],
                "text": s["prompt"],
                "code": s["code"],
                "example": s["test_list"][0],
                "tests": "\n".join(s["test_list"]),
                "setup": "\n".join(s["test_imports"]),
            })
    else:
        data = load_dataset("evalplus/mbppplus")["test"]
        for s in data:
            # remove any assert lines from the prompt text
            lines = s["prompt"].replace('"""', "").splitlines()
            text = "\n".join(l for l in lines if not l.strip().startswith("assert")).strip()
            problems.append({
                "task_id": s["task_id"],
                "text": text,
                "code": s["code"],
                "example": s["test_list"][0],
                "tests": s["test"],
                "setup": "\n".join(s["test_imports"]),
            })
    return problems


problems = load_problems(DATASET, SPLIT)[:N_PROBLEMS]
print(len(problems), "problems")
print(problems[0]["text"])

## Check the reference solutions
Should print `(True, None)` for each problem. If not, the data loading is wrong.

In [ ]:
for p in problems:
    print(p["task_id"], evaluate_final(p["code"], p["tests"], setup_code=p["setup"]))

## Quick test on one problem

In [ ]:
problem = problems[0]
print(problem["text"])

if METHOD == "eig":
    result = run_clarification_pipeline(problem, verbose=True)
    print("\nFinal weights:", [round(w, 3) for w in result["weights"]])
else:
    result = run_one_shot(problem)

print("Questions asked:", result["questions_asked"])
print("Tokens used:", result["tokens"])
print("\nFinal code:\n", result["final_code"])
print(evaluate_final(result["final_code"], problem["tests"], setup_code=problem["setup"]))

## Run experiments
Each problem is saved to `results/` as soon as it finishes.
If the run stops, run the cells again and it continues where it stopped.

In [ ]:
RUN_SPLIT = "test"
RUN_N_PROBLEMS = 25
model_tag = "3b" if "3B" in MODEL_NAME else "1.5b"
os.makedirs("results", exist_ok=True)


# run one dataset + method and save each problem to results/
def run_experiment(dataset, method):
    run_name = f"{dataset}_{model_tag}_{method}_{RUN_SPLIT}"
    if not SHOW_EXAMPLE:
        run_name += "_noexample"
    print("\n=====", run_name, "=====")

    problems = load_problems(dataset, RUN_SPLIT)[:RUN_N_PROBLEMS]
    out_path = f"results/{run_name}.jsonl"

    # skip problems that are already done
    done = set()
    if os.path.exists(out_path):
        with open(out_path) as f:
            for line in f:
                done.add(json.loads(line)["task_id"])
    print("already done:", len(done))

    # save the settings of this run
    with open(f"results/{run_name}_config.json", "w") as f:
        json.dump({"model": MODEL_NAME, "dataset": dataset, "split": RUN_SPLIT, "method": method,
                   "n": RUN_N_PROBLEMS, "SHOW_EXAMPLE": SHOW_EXAMPLE,
                   "N_CANDIDATES": N_CANDIDATES, "M_TESTS": M_TESTS, "K_MAX": K_MAX, "EPS": EPS,
                   "GAMMA": GAMMA, "ETA": ETA, "LAMBDA": LAMBDA, "RHO": RHO,
                   "MIN_COVERAGE": MIN_COVERAGE}, f, indent=2)

    for problem in tqdm(problems):
        if problem["task_id"] in done:
            continue
        torch.manual_seed(problem["task_id"])   # same seed per problem, so runs repeat

        try:
            if method == "eig":
                result = run_clarification_pipeline(problem)
            else:
                result = run_one_shot(problem)
            success, error = evaluate_final(result["final_code"], problem["tests"], setup_code=problem["setup"])
        except Exception as e:
            result = {"final_code": None, "questions_asked": 0, "tokens": 0}
            success, error = False, f"pipeline error: {e}"

        record = {
            "task_id": problem["task_id"], "passed": success, "error": error,
            "questions_asked": result["questions_asked"], "tokens": result["tokens"],
            "final_code": result["final_code"],
        }
        with open(out_path, "a") as f:
            f.write(json.dumps(record) + "\n")

### MBPP
1.5B: one-shot and EIG. 3B: EIG only.

In [ ]:
if "3B" in MODEL_NAME:
    run_experiment("mbpp", "eig")
else:
    run_experiment("mbpp", "oneshot")
    run_experiment("mbpp", "eig")

### MBPP+
Run with the 3B model only.

In [ ]:
if "3B" in MODEL_NAME:
    run_experiment("mbppplus", "eig")
else:
    print("MBPP+ uses the 3B model. Change MODEL_NAME in generate.py and restart.")

## Results
Reads every run in `results/` and saves a table to `results/summary.md`.
Only the first 25 problems of each run are used, so all runs are compared on the same problems.
This cell does not need the model, so after Setup you can run it directly.

In [ ]:
import os, json, math

MAX_PROBLEMS = 25


# pass@1, 95% interval and averages for one run
def get_stats(run_name):
    with open(f"results/{run_name}.jsonl") as f:
        records = [json.loads(line) for line in f]
    records = records[:MAX_PROBLEMS]
    n = len(records)
    passed = sum(1 for r in records if r["passed"])
    p = passed / n
    margin = 1.96 * math.sqrt(p * (1 - p) / n)
    return {
        "run": run_name, "passed": passed, "n": n, "pass_at_1": p,
        "low": max(0.0, p - margin), "high": min(1.0, p + margin),   # keep inside [0, 1]
        "avg_q": sum(r["questions_asked"] for r in records) / n,
        "avg_tokens": sum(r["tokens"] for r in records) / n,
    }


runs = sorted(f[:-len(".jsonl")] for f in os.listdir("results") if f.endswith(".jsonl"))
rows = []
for run in runs:
    s = get_stats(run)
    rows.append(s)
    print(f"{run}: passed {s['passed']}/{s['n']}")
    print(f"  pass@1 = {s['pass_at_1']:.3f}  [{s['low']:.3f}, {s['high']:.3f}]")
    print(f"  avg questions = {s['avg_q']:.2f}, avg tokens = {s['avg_tokens']:.0f}")

# save a markdown table
lines = ["| Run | Passed | pass@1 | 95% CI | Avg. questions | Avg. tokens |",
         "|---|---|---|---|---|---|"]
for s in rows:
    lines.append(f"| {s['run']} | {s['passed']}/{s['n']} | {s['pass_at_1']:.2f} | "
                 f"[{s['low']:.2f}, {s['high']:.2f}] | {s['avg_q']:.2f} | {s['avg_tokens']:.0f} |")
with open("results/summary.md", "w") as f:
    f.write("\n".join(lines) + "\n")
print("\nsaved results/summary.md")

## One-shot vs EIG on the same problems
Shows which problems EIG fixed or broke compared to one-shot.

In [ ]:
# passed / failed for each task in a run
def load_passed(name):
    passed = {}
    with open(f"results/{name}.jsonl") as f:
        for line in f:
            r = json.loads(line)
            passed[r["task_id"]] = r["passed"]
    return passed


one = load_passed("mbpp_1.5b_oneshot_test")
eig = load_passed("mbpp_1.5b_eig_test")
common = [t for t in eig if t in one]

print(f"shared problems: {len(common)}")
print(f"one-shot pass@1: {sum(one[t] for t in common) / len(common):.2f}")
print(f"EIG pass@1:      {sum(eig[t] for t in common) / len(common):.2f}")
print("EIG fixed:", [t for t in common if eig[t] and not one[t]])
print("EIG broke:", [t for t in common if one[t] and not eig[t]])

## Summary

All runs use the first 25 problems of each dataset (MBPP sanitized test split, MBPP+).

| Model | Dataset | Method | pass@1 | Lower bound (95%) | Avg. questions | Avg. tokens |
|---|---|---|---|---|---|---|
| 1.5B | MBPP | One-shot | 0.68 | 0.50 | 0.00 | 171 |
| 1.5B | MBPP | EIG | 0.80 | 0.64 | 0.60 | 2628 |
| 3B | MBPP | EIG | 0.84 | 0.70 | 0.56 | 3076 |
| 3B | MBPP+ | EIG | 0.88 | 0.75 | 0.56 | 2994 |

- On the same 25 problems, EIG fixed 3 problems that one-shot failed and broke none.
- EIG asked no question on 13 problems (candidates agreed) and all 13 passed.
- EIG uses about 15x more tokens than one-shot.
- MBPP+ scores higher because its first 25 problems are a different set with some easy ones. On the 18 shared problems, 3B passed 16 on both.

**Limitations:** only 25 problems (wide intervals), no one-shot run for 3B, small models and 8 candidates (paper uses 16).